# Exploration et analyse du catalogue Netflix.

In [9]:
import pandas as pd

In [10]:
netflix = pd.read_csv('../data/raw/netflix_titles_raw.csv')

## Exploration du dataset

- Le nombre de colonnes
- Les noms de colonnes
- Le compte des valeurs manquantes
- Les dtypes par colonnes
- La taille du jeu de données
- Le nombre de cellules

### Informations sur le dataset

In [12]:
netflix.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


### Description des statistiques courantes du dataset

In [21]:
netflix.describe()

,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


### La taille, le nombre de cellules et les dtypes du dataset

In [22]:
netflix.shape

(8807, 12)

In [23]:
netflix.size

105684

In [24]:
netflix.dtypes

show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

### Les valeurs manquantes

- Détecter les valeurs manquantes
- Compter le nombre de valeurs manquantes
- Calculer le pourcentage de valeurs manquantes

In [20]:
netflix.isna().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [19]:
netflix.isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.299080
cast            0.093675
country         0.094357
date_added      0.001135
release_year    0.000000
rating          0.000454
duration        0.000341
listed_in       0.000000
description     0.000000
dtype: float64

#### Hypothèse

Est-ce que `director` est vide parce que la donnée n'a jamais été collectée (MCAR — manquant complètement au hasard), ou parce qu'il existe une vraie raison structurelle (par exemple : les TV Shows n'ont souvent pas de "réalisateur" unique désigné, contrairement aux films) ?

On distingue généralement trois types de données manquantes :

- MCAR (Missing Completely At Random) : la probabilité d'absence n'a aucun lien avec les données elles-mêmes ou d'autres variables (Rare en pratique).

- MAR (Missing At Random) : l'absence est liée à une autre colonne observable (ex : director manquant plus souvent quand type == "TV Show").

- MNAR (Missing Not At Random) : l'absence est liée à la valeur elle-même, non observable (ex : les gens à très haut revenu qui refusent de déclarer leur salaire).

> Pourquoi ça compte concrètement : si tu supprimes ou imputes sans comprendre le mécanisme, tu peux introduire un biais silencieux dans tes données. Si `director` manque majoritairement pour les séries TV, et que tu drop toutes les lignes avec director manquant, tu viens de supprimer presque toutes tes séries TV du dataset — sans le vouloir.

In [71]:
netflix[netflix['type'] == 'TV Show'].isna().sum()

show_id            0
type               0
title              0
director        2446
cast             350
country          391
date_added        10
release_year       0
rating             2
duration           0
listed_in          0
description        0
dtype: int64

In [72]:
netflix[netflix['type'] == 'TV Show'].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.914051
cast            0.130792
country         0.146114
date_added      0.003737
release_year    0.000000
rating          0.000747
duration        0.000000
listed_in       0.000000
description     0.000000
dtype: float64

##### Pour compléter la vérification de l'hypothèse

Taux conditionnel P(manquant | TV Show) vs Proportion P(TV Show | manquant)).

Avant la conclusion définitivement, faire le même calcul côté Movie sur director uniquement. Si le taux est très bas pour Movie et très haut pour TV Show, alors l'hypothèse MAR est solidement confirmée — c'est le contraste entre les deux groupes qui fait la preuve, pas juste un chiffre isolé.

In [76]:
netflix[netflix['type'] == 'Movie'].isna().sum()

show_id           0
type              0
title             0
director        188
cast            475
country         440
date_added        0
release_year      0
rating            2
duration          3
listed_in         0
description       0
dtype: int64

In [77]:
netflix[netflix['type'] == 'Movie'].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.030664
cast            0.077475
country         0.071766
date_added      0.000000
release_year    0.000000
rating          0.000326
duration        0.000489
listed_in       0.000000
description     0.000000
dtype: float64

Sur le contraste director — 3,07% pour Movie vs 91,4% pour TV Show. Ça, c'est une preuve solide et propre d'un mécanisme MAR : l'absence de director est quasi exclusivement structurelle (les séries n'ont généralement pas de réalisateur unique crédité), pas aléatoire. Tu peux documenter cette conclusion telle quelle dans ton rapport final, avec les deux chiffres côte à côte — c'est exactement le type de justification qu'on attend.

#### Hypothèse

Est ce que `cast` est vide parce que les documentaires et docuseries n'ont pas de casting ?


In [100]:
netflix[netflix['listed_in'].str.contains('Documentaries')].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.065593
cast            0.487917
country         0.086306
date_added      0.000000
release_year    0.000000
rating          0.000000
duration        0.000000
listed_in       0.000000
description     0.000000
dtype: float64

In [101]:
netflix[netflix['listed_in'].str.contains('Documentaries')].isna().sum()

show_id           0
type              0
title             0
director         57
cast            424
country          75
date_added        0
release_year      0
rating            0
duration          0
listed_in         0
description       0
dtype: int64

In [99]:
netflix[netflix['listed_in'].str.contains('Docuseries')].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.848101
cast            0.524051
country         0.164557
date_added      0.002532
release_year    0.000000
rating          0.000000
duration        0.000000
listed_in       0.000000
description     0.000000
dtype: float64

In [103]:
netflix[netflix['listed_in'].str.contains('Docuseries')].isna().sum()

show_id           0
type              0
title             0
director        335
cast            207
country          65
date_added        1
release_year      0
rating            0
duration          0
listed_in         0
description       0
dtype: int64

Si ce nombre est 0, ton addition simple était correcte. S'il est différent de 0, il faut le soustraire de ta somme pour ne pas compter ces lignes deux fois (principe d'inclusion-exclusion : `|A ∪ B| = |A| + |B| - |A ∩ B|`).

In [ ]:
424 + 207 + 0

: 

##### Pour compléter la vérification de l'hypothèse

Hypothèse MAR confirmée à 76%, le reste (24%) reste sans explication claire pour l'instant, ce qui est normal

In [ ]:
netflix[
    netflix['listed_in'].str.contains('Documentaries') &
    netflix['listed_in'].str.contains('Docuseries')
].shape[0]

0

In [116]:
631/825

0.7648484848484849

In [93]:
netflix.head(20)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
5,s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, H...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",The arrival of a charismatic young priest brin...
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",NaN,"September 24, 2021",2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s..."
8,s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,"September 24, 2021",2021,TV-14,9 Seasons,"British TV Shows, Reality TV",A talented batch of amateur bakers face off in...
9,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...",United States,"September 24, 2021",2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...
